In [5]:
from src.config import LLMParams

In [28]:
import asyncio
import json
import logging

import numpy as np
import random

import pandas as pd
from numpy.random import RandomState
from pydantic import TypeAdapter

from src.config import AppSettings, LLMConfig
from src.schemas import (
    PromptLogprob,
    PtbScenario,
    PtbScenarioRes,
    TokenEntropy,
    WordInfo,
    WordInfoRes,
)
from src.utils.base import configure_logging, create_openai_client
from src.utils.perturb import get_words_and_indices

logger = logging.getLogger("research")

In [35]:
model = "Qwen/Qwen3-4B"
seed = 42
samples = 10

In [9]:
with open("../config_colab.json", "r") as f:
    config = AppSettings.model_validate_json(f.read())

In [14]:
config.logging_conf_file='../logging_config.json'

In [15]:
configure_logging(config=config)

In [33]:
df = pd.read_json("../MuSeRC/train.jsonl", lines=True)
small_df = df.sample(n=samples, random_state=RandomState(seed=seed))
del df

small_list = small_df.to_dict(orient="records")
ta = TypeAdapter(list[ReadingComprehensionItem])
items = ta.validate_python(small_list)

client = create_openai_client(config=config.llm)
semaphore = asyncio.Semaphore(config.llm.async_cals)

In [34]:
item = items[0]

random.seed(a=seed)
context = item.passage.text
question = random.choice(item.passage.questions)
answer = random.choice(question.answers)


scenario={
             "name": f"Запрос 0",
             "context": context,
             "reference": answer.text,
             "question": question.question,
         }

words_infos: list[WordInfo] = get_words_and_indices(scenario["context"])

In [30]:
words_infos

[{'word': '1', 'start': 1, 'end': 2},
 {'word': 'Спустя', 'start': 4, 'end': 10},
 {'word': 'какое', 'start': 11, 'end': 16},
 {'word': 'то', 'start': 17, 'end': 19},
 {'word': 'время', 'start': 20, 'end': 25},
 {'word': 'везение', 'start': 26, 'end': 33},
 {'word': 'прекращается', 'start': 34, 'end': 46},
 {'word': '2', 'start': 49, 'end': 50},
 {'word': 'Редакции', 'start': 52, 'end': 60},
 {'word': 'наперебой', 'start': 61, 'end': 70},
 {'word': 'стараются', 'start': 71, 'end': 80},
 {'word': 'обжулить', 'start': 81, 'end': 89},
 {'word': 'Мартина', 'start': 90, 'end': 97},
 {'word': '3', 'start': 100, 'end': 101},
 {'word': 'Добыть', 'start': 103, 'end': 109},
 {'word': 'у', 'start': 110, 'end': 111},
 {'word': 'них', 'start': 112, 'end': 115},
 {'word': 'деньги', 'start': 116, 'end': 122},
 {'word': 'за', 'start': 123, 'end': 125},
 {'word': 'публикации', 'start': 126, 'end': 136},
 {'word': 'оказывается', 'start': 137, 'end': 148},
 {'word': 'нелёгким', 'start': 149, 'end': 157},

In [ ]:
text = "context: " + scenario["context"] + "\nquestion: " + scenario["question"]

_, tokens_result = analyze_prompt_entropy(
    idx=str(item.idx),
    scenario={"name": f"Запрос 0", "text": text},
    client=client,
    semaphore=semaphore,
    model=model,
    config=config.llm,
)